# COMP64702 — RAG Coursework: Main Inference Pipeline

**Course:** COMP64702 Transforming Text Into Meaning  
**Cuisine:** Mediterranean  
**Authors:** Person 1, Person 2, Person 3

---

## What This Notebook Does

This is the **master pipeline notebook** (Deliverable 3). It runs the complete RAG inference pipeline end-to-end:

1. Loads pre-computed artifacts from the experimentation phase (chunks, embeddings)
2. Builds the FAISS index and BM25 index for hybrid retrieval
3. Loads the Qwen/Qwen2.5-0.5B-Instruct language model
4. Reads an input JSON file containing questions
5. For each question: retrieves top-5 chunks → generates an answer using the instructed prompt
6. Saves `test_outputs.json` in the required format (Deliverable 5)
7. **(Optional)** Evaluates generated answers against gold-standard answers if provided

## Selected Configuration

The best configuration was determined through systematic experimentation documented in `experiments.ipynb`:

| Component | Selection | Justification |
|---|---|---|
| Chunking | Overlapping (200 tokens, 50 overlap) | Highest Hit@5 — solves boundary-splitting |
| Embedding | BAAI/bge-small-en-v1.5 | Highest Hit@1 and MRR — retrieval-optimised |
| Retrieval | Hybrid (dense=0.6, BM25=0.4) | Best combined score across 6 alpha settings |
| Prompting | Instructed prompt | Highest ROUGE-L — reduces hallucination |
| Max new tokens | 150 | Sufficient for detailed answers |

## Required Input Files

| File | Source | Description |
|---|---|---|
| `chunks/chunks_overlap.json` | Generated by `experiments.ipynb` | Selected overlapping chunks |
| `embeddings/embeddings_bge.pkl` | Generated by `experiments.ipynb` | BGE embeddings + metadata |
| `input_payload.json` | Provided on demo day (or custom) | Questions to answer |
| `gold_answers.json` | *(Optional)* Provided on demo day | Gold-standard answers for evaluation |

## Input / Output Format

**Input** (`input_payload.json`):
```json
{
  "queries": [
    {"query_id": "0", "query": "What is hummus made from?"},
    {"query_id": "1", "query": "Where did paella originate?"}
  ]
}
```

**Output** (`test_outputs.json`):
```json
{
  "results": [
    {
      "query_id": "0",
      "query": "What is hummus made from?",
      "response": "Hummus is made from chickpeas...",
      "retrieved_context": [
        {"doc_id": "000", "text": "..."},
        {"doc_id": "001", "text": "..."}
      ]
    }
  ]
}
```

---

## 0. Configuration

**Change `INPUT_FILE` below to point to the query file provided on demo day.** All other paths should work without modification if the project folder structure is intact.

If gold-standard answers are provided for evaluation, set `GOLD_FILE` to the path of that file.

In [ ]:
# Path to gold answers (provided during live demo for evaluation)
# Leave empty string '' until demo day — set path when gold answers are provided
GOLD_FILE   = ''

In [ ]:
# Path to input queries (change this on demo day)
INPUT_FILE  = 'ea_test_v2.json'

In [ ]:


# Path to output file (Deliverable 5)
OUTPUT_FILE = 'test_outputs.json'

In [ ]:
#  ╔══════════════════════════════════════════════════════════════╗
#  ║  CONFIGURATION — Edit this cell before running             ║
#  ╚══════════════════════════════════════════════════════════════╝





# Pre-computed artifacts from experiments.ipynb
CHUNKS_FILE     = 'chunks_overlap.json'
EMBEDDINGS_FILE = 'embeddings_bge.pkl'

# Model and retrieval settings (do not change — these match experiments)
LLM_MODEL       = 'Qwen/Qwen2.5-0.5B-Instruct'
EMBED_MODEL     = 'BAAI/bge-small-en-v1.5'
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
HYBRID_ALPHA    = 0.6       # dense weight (BM25 weight = 0.4)
TOP_K           = 5         # max chunks per query (spec requirement)
MAX_NEW_TOKENS  = 150       # generation length

---

## 1. Install Dependencies

In [ ]:
!pip install sentence-transformers faiss-cpu rank_bm25 transformers accelerate sentencepiece rouge-score bert-score --quiet
print('Dependencies ready.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
Dependencies ready.


---

## 2. Imports

In [ ]:
import json
import pickle
import re
from pathlib import Path

import numpy as np
import faiss
import torch
from rank_bm25             import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers          import AutoTokenizer, AutoModelForCausalLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


---

## 3. Load Pre-Computed Artifacts

These files were generated by `experiments.ipynb`. The chunks contain the overlapping-chunked corpus and the embeddings contain the BGE vector representations.

In [ ]:
# ── Load chunks ───────────────────────────────────────────────
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

chunk_texts   = [c['text']     for c in chunks]
chunk_ids     = [c['chunk_id'] for c in chunks]
chunk_sources = [c['source']   for c in chunks]
print(f'Loaded {len(chunks):,} chunks')


# ── Load embeddings ───────────────────────────────────────────
with open(EMBEDDINGS_FILE, 'rb') as f:
    emb_data = pickle.load(f)

embeddings = emb_data['embeddings']
print(f'Loaded embeddings: shape {embeddings.shape}')
print(f'Embedding model: {emb_data["model_name"]}')

Loaded 161 chunks
Loaded embeddings: shape (161, 384)
Embedding model: BAAI/bge-small-en-v1.5


---

## 4. Build Retrieval Indices

Two indices are built for hybrid retrieval:
- **FAISS** — inner product search over L2-normalised BGE embeddings (= cosine similarity)
- **BM25** — keyword frequency matching using tokenised chunk text

In [ ]:
# ── FAISS index ───────────────────────────────────────────────
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
print(f'FAISS index: {faiss_index.ntotal:,} vectors')


# ── BM25 index ────────────────────────────────────────────────
def bm25_tokenise(text):
    return re.findall(r'\b\w+\b', text.lower())

bm25_corpus = [bm25_tokenise(c['text']) for c in chunks]
bm25_index  = BM25Okapi(bm25_corpus)
print(f'BM25 index: {len(bm25_corpus):,} chunks')


# ── Embedding model for query encoding ────────────────────────
print(f'Loading embedding model: {EMBED_MODEL}')
embed_model = SentenceTransformer(EMBED_MODEL)
print('Embedding model ready.')

FAISS index: 161 vectors
BM25 index: 161 chunks
Loading embedding model: BAAI/bge-small-en-v1.5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


---

## 5. Load Language Model

The LLM is `Qwen/Qwen2.5-0.5B-Instruct`, fixed by the coursework specification. We use its chat template for proper prompt formatting.

In [ ]:
print(f'Loading {LLM_MODEL}...')
print('(~1GB download on first run, then cached)')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
)

if DEVICE == 'cpu':
    llm = llm.to(DEVICE)

llm.eval()
print(f'LLM loaded on {next(llm.parameters()).device}')

Loading Qwen/Qwen2.5-0.5B-Instruct...
(~1GB download on first run, then cached)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded on cuda:0


---

## 6. Hybrid Retrieval Function

Hybrid retrieval combines dense (cosine) and sparse (BM25) scores with min-max normalisation:

```
hybrid_score = 0.6 × norm(dense_score) + 0.4 × norm(bm25_score)
```

In [ ]:
def min_max_norm(scores):
    """Normalise scores to [0, 1]."""
    mn, mx = scores.min(), scores.max()
    if mx == mn:
        return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)


def retrieve(question, k=TOP_K):
    """Hybrid retrieval: dense + BM25 with alpha weighting.
    Returns top-k chunks with scores.
    """
    # Dense scores
    q_emb = embed_model.encode(
        [BGE_QUERY_PREFIX + question],
        convert_to_numpy=True, normalize_embeddings=True
    ).astype('float32')[0]
    dense_scores = np.dot(embeddings, q_emb)

    # BM25 scores
    bm25_scores = bm25_index.get_scores(bm25_tokenise(question))

    # Hybrid combination
    hybrid = (HYBRID_ALPHA * min_max_norm(dense_scores)
              + (1 - HYBRID_ALPHA) * min_max_norm(bm25_scores))

    top_k = np.argsort(hybrid)[::-1][:k]

    return [
        {'rank': r + 1,
         'score': float(hybrid[top_k[r]]),
         'chunk_id': chunks[top_k[r]]['chunk_id'],
         'source':   chunks[top_k[r]]['source'],
         'text':     chunks[top_k[r]]['text']}
        for r in range(len(top_k))
    ]

---

## 7. Generation Function

Uses the **instructed prompt** (highest ROUGE-L in experiments) with Qwen chat template. Only newly generated tokens are decoded to avoid prompt echo.

In [ ]:
def build_context(retrieved_chunks):
    """Concatenate retrieved chunks into a context string."""
    return '\n\n'.join(
        f'[Chunk {c["rank"]} | Source: {c["source"]}]\n{c["text"].strip()}'
        for c in retrieved_chunks
    )


def make_instructed_prompt(question, context):
    """Build the instructed prompt (selected strategy)."""
    return (
        f'Answer only from the context provided. '
        f'If the answer is not in the context, say "I don\'t know".\n\n'
        f'Context:\n{context}\n\nQuestion: {question}\n\nAnswer:'
    )


def generate_answer(prompt):
    """Generate answer using Qwen chat template."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt').to(llm.device)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print('Retrieval and generation functions ready.')

Retrieval and generation functions ready.


---

## 8. Load Input Queries

The input file contains the questions to answer. On demo day, replace `INPUT_FILE` in the configuration cell (Section 0) with the path to the file provided by the GTA.

Supported input formats:
- **GTA format** (expected on demo day): `{"queries": [{"query_id": "0", "query": "..."}, ...]}`
- Flat list: `[{"question": "..."}, ...]`
- Benchmark format: `{"sources": [{"source_file": "...", "questions": [...]}]}`

In [ ]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    raw_input = json.load(f)

# Handle different input formats
if isinstance(raw_input, list):
    # Flat list of query objects
    queries = raw_input
elif isinstance(raw_input, dict) and 'queries' in raw_input:
    # GTA input format: {"queries": [{"query_id": "0", "query": "..."}]}
    queries = [{'question': q['query'], 'query_id': q['query_id']} for q in raw_input['queries']]
elif isinstance(raw_input, dict) and 'sources' in raw_input:
    # Benchmark-style nested format — flatten
    queries = []
    for entry in raw_input['sources']:
        for qa in entry['questions']:
            queries.append({'question': qa['question']})
else:
    raise ValueError(f'Unrecognised input format in {INPUT_FILE}')

print(f'Loaded {len(queries)} queries from {INPUT_FILE}')
print(f'First query: {queries[0]["question"][:100]}...')

Loaded 15 queries from ea_test_v2.json
First query: What is the cultural significance of keeping elements separate in Japanese bento boxes and how has t...


---

## 9. Run Inference Pipeline

For each query: retrieve top-5 chunks → build instructed prompt → generate answer.

> **Estimated time:** ~25 minutes on CPU, ~5–8 minutes on Colab T4 GPU.

In [ ]:
import time

results = []
start_time = time.time()

for i, q in enumerate(queries, start=1):
    question = q['question']

    # Step 1: Hybrid retrieval
    retrieved = retrieve(question)

    # Step 2: Build prompt with retrieved context
    context = build_context(retrieved)
    prompt  = make_instructed_prompt(question, context)

    # Step 3: Generate answer
    answer = generate_answer(prompt)

    results.append({
        'query_id': q.get('query_id', str(i-1).zfill(3)),
        'question': question,
        'answer': answer,
        'retrieved_chunks': retrieved,
    })

    if i % 25 == 0 or i == len(queries):
        elapsed = time.time() - start_time
        rate = i / elapsed
        remaining = (len(queries) - i) / rate if rate > 0 else 0
        print(f'  [{i:4d}/{len(queries)}]  {elapsed:.0f}s elapsed  |  ~{remaining:.0f}s remaining')

print(f'\nInference complete: {len(results)} answers generated in {time.time()-start_time:.0f}s')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [  15/15]  48s elapsed  |  ~0s remaining

Inference complete: 15 answers generated in 48s


---

## 10. Save `test_outputs.json` (Deliverable 5)

Output is saved in the required format matching `output_payload_sample.json`:
```json
{
  "results": [
    {
      "query_id": "0",
      "query": "...",
      "response": "...",
      "retrieved_context": [
        {"doc_id": "000", "text": "..."},
        {"doc_id": "001", "text": "..."}
      ]
    }
  ]
}
```

In [ ]:
# Format output in required schema (matches output_payload_sample.json)
test_outputs = {
    "results": [
        {
            "query_id": r['query_id'],
            "query": r['question'],
            "response": r['answer'],
            "retrieved_context": [
                {"doc_id": str(j).zfill(3), "text": c['text']}
                for j, c in enumerate(r['retrieved_chunks'])
            ]
        }
        for r in results
    ]
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(test_outputs, f, indent=2, ensure_ascii=False)

print(f'Saved {len(test_outputs["results"])} answers -> {OUTPUT_FILE}')
print(f'Format check — first entry:')
print(json.dumps(test_outputs['results'][0], indent=2))

Saved 15 answers -> test_outputs.json
Format check — first entry:
{
  "query_id": 0,
  "query": "What is the cultural significance of keeping elements separate in Japanese bento boxes and how has this tradition evolved with the emergence of kyaraben?",
  "response": "Keeping elements separate in Japanese bento boxes is a traditional practice that emphasizes individuality and personalization. This tradition has evolved significantly over time, reflecting changes in societal values and dietary preferences. Initially, bento boxes were primarily used for convenience and to save space on tables, but they have since become more sophisticated, incorporating various elements to create unique experiences.\n\nAs society became more aware of the importance of individuality and self-expression, bento boxes began to incorporate more distinct elements. For instance:\n\n1. **Personalized Items**: Bento boxes now feature items that reflect the individual tastes and preferences of each customer. This c

---

## 11. Output Verification

Quick sanity checks to confirm the output matches the required format before submission.

In [ ]:
# ── Verify output file ─────────────────────────────────────────
with open(OUTPUT_FILE, 'r') as f:
    check = json.load(f)

entries = check['results']
print(f'Total entries: {len(entries)}')
print(f'Expected:      {len(queries)}')
print(f'Match:         {"YES" if len(entries) == len(queries) else "NO — MISMATCH"}')
print()

# Check all required fields are present
required_fields = {'query_id', 'query', 'response', 'retrieved_context'}
for i, entry in enumerate(entries):
    missing = required_fields - set(entry.keys())
    if missing:
        print(f'ERROR: Entry {i} missing fields: {missing}')
        break
else:
    print('All entries have required fields (query_id, query, response, retrieved_context).')
print()

# Check no empty responses
empty = [i for i, e in enumerate(entries) if not e['response'].strip()]
if empty:
    print(f'WARNING: {len(empty)} entries have empty responses: {empty[:5]}')
else:
    print('No empty responses.')
print()

# Check retrieved_context has chunks
no_ctx = [i for i, e in enumerate(entries) if len(e['retrieved_context']) == 0]
if no_ctx:
    print(f'WARNING: {len(no_ctx)} entries have no retrieved context')
else:
    print(f'All entries have retrieved context (top-{TOP_K} chunks each).')
print()

# Preview
print('Sample outputs:')
for entry in entries[:3]:
    print(f'  [query_id={entry["query_id"]}] Q: {entry["query"][:80]}...')
    print(f'         A: {entry["response"][:100]}')
    print(f'         Chunks: {len(entry["retrieved_context"])}')
    print()

Total entries: 15
Expected:      15
Match:         YES

All entries have required fields (query_id, query, response, retrieved_context).

No empty responses.

All entries have retrieved context (top-5 chunks each).

Sample outputs:
  [query_id=0] Q: What is the cultural significance of keeping elements separate in Japanese bento...
         A: Keeping elements separate in Japanese bento boxes is a traditional practice that emphasizes individu
         Chunks: 5

  [query_id=1] Q: At what temperature is sushi rice traditionally served and what effect does cool...
         A: Sushi rice is traditionally served at room temperature. Cooling sushi rice can significantly affect 
         Chunks: 5

  [query_id=2] Q: What is the annual consumption of kimchi in modern South Korea and what incident...
         A: According to the information provided, the annual consumption of kimchi in modern South Korea was es
         Chunks: 5



---

## 12. Evaluation Pipeline (Live Demo)

If gold-standard answers are provided (e.g. during the live demo), this section evaluates the generated responses using three metrics:

| Metric | What It Measures |
|---|---|
| Exact Match | Gold answer appears (case-insensitive substring) in the generated response |
| ROUGE-L | Longest common subsequence overlap — measures lexical similarity to gold answer |
| BERTScore F1 | Semantic similarity via BERT embeddings — handles paraphrasing and synonyms |

**Set `GOLD_FILE` in the configuration cell (Section 0) to the path of the gold answers file before running this section.** If `GOLD_FILE` is left empty, this section is skipped.

In [ ]:
if GOLD_FILE:
    import pandas as pd
    from rouge_score import rouge_scorer
    from bert_score import score as bertscore_score
    from IPython.display import display

    # ── Load gold answers ─────────────────────────────────────
    with open(GOLD_FILE, 'r', encoding='utf-8') as f:
        gold_data = json.load(f)

    # Build lookup: query_id -> gold answer
    # Handle multiple possible formats the GTA might use
    if isinstance(gold_data, dict) and 'results' in gold_data:
        gold_lookup = {
            str(g['query_id']): g.get('response', g.get('answer', ''))
            for g in gold_data['results']
        }
    elif isinstance(gold_data, dict) and 'queries' in gold_data:
        gold_lookup = {
            str(g['query_id']): g.get('answer', g.get('response', ''))
            for g in gold_data['queries']
        }
    elif isinstance(gold_data, list):
        gold_lookup = {
            str(g.get('query_id', i)): g.get('answer', g.get('response', ''))
            for i, g in enumerate(gold_data)
        }
    else:
        raise ValueError(f'Unrecognised gold answer format in {GOLD_FILE}')

    print(f'Loaded {len(gold_lookup)} gold answers from {GOLD_FILE}')

    # ── Match predictions to gold ─────────────────────────────
    preds, golds, matched_ids = [], [], []
    for r in results:
        qid = str(r['query_id'])
        if qid in gold_lookup:
            preds.append(r['answer'])
            golds.append(gold_lookup[qid])
            matched_ids.append(qid)

    print(f'Matched {len(preds)}/{len(results)} predictions to gold answers')
    print()

    # ── Exact Match ───────────────────────────────────────────
    em = [1 if g.lower().strip() in p.lower() else 0 for p, g in zip(preds, golds)]

    # ── ROUGE-L ───────────────────────────────────────────────
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge = [scorer.score(g, p)['rougeL'].fmeasure for p, g in zip(preds, golds)]

    # ── BERTScore ─────────────────────────────────────────────
    print('Computing BERTScore...')
    _, _, F1 = bertscore_score(preds, golds, lang='en', verbose=False)
    bert_f1 = [float(f) for f in F1]

    # ── Summary table ─────────────────────────────────────────
    summary = pd.DataFrame([
        {'Metric': 'Exact Match',  'Score': round(float(np.mean(em)), 3)},
        {'Metric': 'ROUGE-L',      'Score': round(float(np.mean(rouge)), 3)},
        {'Metric': 'BERTScore F1', 'Score': round(float(np.mean(bert_f1)), 3)},
    ])

    print('\nEvaluation Results:')
    display(summary)

    # ── Save evaluation results ───────────────────────────────
    eval_output = {
        'num_evaluated': len(preds),
        'exact_match':   round(float(np.mean(em)), 3),
        'rouge_l':       round(float(np.mean(rouge)), 3),
        'bertscore_f1':  round(float(np.mean(bert_f1)), 3),
    }
    with open('evaluation_results.json', 'w') as f:
        json.dump(eval_output, f, indent=2)
    print(f'\nSaved evaluation_results.json')

else:
    print('No GOLD_FILE set — skipping evaluation.')
    print('Set GOLD_FILE in the configuration cell (Section 0) to run evaluation.')